# Before the Signal Dies: XGBoost outage-report model (positive–unlabelled)

This notebook trains and tests an **XGBoost** model on the Optus significant-outage register. It is the learned counterpart to the transparent index in `susceptibility_index.ipynb`.

**Target:** `reported_outage_label`

- `1` = the locality appears in a WA-only significant-outage report (13 Dec 2025 – 4 Oct 2026). There are 114 such localities.
- Blank = **unknown** (1,585 localities). There are no confirmed outage-free localities.

This is **positive–unlabelled (PU)** data. The model learns to separate reported localities from unknown ones. Its rankings match true outage susceptibility only if reported outages are a random sample of all outages. That assumption is called **SCAR** (selected completely at random), and it is probably false here. The register lists "significant" outages, which need many affected services, so populated places are far more likely to appear.

**What the model can tell us:** where reported outages concentrate, and whether that pattern holds for incidents it has not seen.
**What it cannot tell us:** outage probabilities, accuracy, or false-alarm rates.

**Inputs:** the 11 locality features in the allowlist in `outage_data_quality.json`. Fire and cyclone history ends **before** the outage window, so no future information leaks in.

## 0. Environment setup on LinuxONE

The model needs `xgboost`, `pandas` and `numpy`. `matplotlib` is optional. scikit-learn is **not** needed: the splits, metrics and logistic baseline are written with numpy.

If the import cell below fails, open a terminal (**File → New → Terminal**) and run:

```bash
python3 -m venv ~/haxorus-venv --system-site-packages
source ~/haxorus-venv/bin/activate
pip install xgboost ipykernel
python -m ipykernel install --user --name haxorus --display-name "Python (haxorus)"
```

Then choose the **Python (haxorus)** kernel (**Kernel → Change Kernel**).

LinuxONE uses the s390x processor, and PyPI may not have a ready-made XGBoost package for it. If pip starts compiling and fails, install the build tools first (`sudo apt install -y cmake g++`) and retry.

## 1. Setup and configuration

In [ ]:
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import xgboost as xgb
except ImportError as exc:
    raise ImportError(
        f"XGBoost is not installed for this kernel ({sys.executable}). Follow the setup steps in section 0."
    ) from exc

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

print("Python    ", sys.version.split()[0], "on", platform.machine())
print("Kernel    ", sys.executable)
print("numpy     ", np.__version__)
print("pandas    ", pd.__version__)
print("xgboost   ", xgb.__version__)


def find_repo_root(start=None):
    """Walk up from the working directory until data/processed is found."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError("Could not find data/processed above " + str(here))


ROOT = find_repo_root()
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
print("Repository root:", ROOT)

In [ ]:
# Repeat every test with several random seeds; results are reported as mean and range.
SEEDS = [0, 1, 2, 3, 4]

# Unreported localities set aside as the fixed comparison set, chosen by whole grid cells.
REFERENCE_SHARE = 0.20
GRID_DEGREES = 1.0

# "Captured" means a held-out reported locality scores in the top 20% of the
# comparison set - the same share as the High band.
HIGH_SHARE = 0.20
MEDIUM_SHARE = 0.30

# The 31 May - 1 Jun 2026 storm: weather-related WA-only reports starting on these WA dates.
STORM_DATES_WA = ["2026-05-31", "2026-06-01"]

# Uses XGBoost's native API (xgb.train), which unlike XGBClassifier does not need scikit-learn.
# Settings are fixed, not tuned: with so few independent incidents, tuning would mostly fit noise.

# First version, kept for comparison. It memorised exact population values of a few
# reported localities (repeated splits at 112-117 residents).
XGB_PARAMS_ORIGINAL = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "eta": 0.05,
    "max_depth": 3,
    "min_child_weight": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "lambda": 1.0,
    "tree_method": "hist",
    "nthread": 4,
}
NUM_BOOST_ROUND_ORIGINAL = 300

# Regularised version used for the final model: shallower trees, more evidence needed
# per split, stronger shrinkage of leaf values, fewer rounds, plus monotone constraints.
XGB_PARAMS = {
    **XGB_PARAMS_ORIGINAL,
    "max_depth": 2,           # 3 -> 2: each tree combines at most two features
    "min_child_weight": 10,   # 3 -> 10: a split needs far more (weighted) localities on each side
    "lambda": 5.0,            # 1 -> 5: shrinks extreme leaf values
    "gamma": 1.0,             # 0 -> 1: a split must improve the fit by a minimum amount
}
NUM_BOOST_ROUND = 150         # 300 -> 150

# Monotone constraints: +1 = a higher value can never lower the score,
# -1 = a higher value can never raise it. Features not listed are unconstrained
# (site counts and densities are genuinely non-linear: few sites score high,
# very dense CBD coverage scores lower).
MONOTONE = {
    "population_2021": 1,                            # more customers -> more reportable
    "nearest_optus_site_km": -1,                     # closer site -> more reportable Optus outage
    "recorded_bushfire_count_before_window": 1,
    "recorded_bushfires_per_year_before_window": 1,
    "nearby_cyclone_track_count_before_window": 1,
}

## 2. Load and validate the data

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


INPUTS = {
    "modelling": PROCESSED / "wa_outage_modelling.csv",
    "outages": PROCESSED / "wa_outages_clean.csv",
    "features": PROCESSED / "wa_locality_features.csv",
    "quality": PROCESSED / "outage_data_quality.json",
}
data = pd.read_csv(INPUTS["modelling"], dtype={"sal_code_2021": str})
outages = pd.read_csv(INPUTS["outages"], dtype={"event_id": str})
snapshot = pd.read_csv(INPUTS["features"], dtype={"sal_code_2021": str})
quality = json.loads(INPUTS["quality"].read_text())

FEATURES = quality["model_features"]
TARGET = quality["target_column"]

for name in ["wa_outage_modelling.csv", "wa_outages_clean.csv", "wa_locality_features.csv"]:
    key = {"wa_outage_modelling.csv": "modelling", "wa_outages_clean.csv": "outages",
           "wa_locality_features.csv": "features"}[name]
    ok = quality["outputs"][name]["sha256"] == sha256(INPUTS[key])
    print(f"{name}: {'matches' if ok else 'DIFFERS FROM'} the checksum in outage_data_quality.json")

assert data["sal_code_2021"].is_unique
assert data[FEATURES].notna().all().all(), "Blank model features"
assert set(data[TARGET].dropna().unique()) <= {1}, "Target must be 1 or blank"

data = data.sort_values("sal_code_2021").reset_index(drop=True)
data["y"] = data[TARGET].fillna(0).astype(int)
print(f"{len(data)} localities: {data['y'].sum()} reported, {(data['y'] == 0).sum()} unknown")
print("Model features:", FEATURES)

## 3. Incident groups and the comparison set

**Incident groups.** Localities in the same outage report must stay on the same side of a split. We also link localities that appear in several reports, so incidents sharing a locality merge into one group. All storm reports from 31 May – 1 Jun 2026 are treated as one incident.

**Comparison set.** About 20% of unreported localities are set aside, chosen by whole 1° grid cells so they are not simply neighbours of training localities. No model ever trains on them. Every held-out reported locality is ranked against this same set.

In [ ]:
storm_events = set(outages.loc[
    (outages["weather_related_diagnostic_only"] == 1)
    & (outages["geographic_scope"] == "wa_only")
    & (outages["start_date_wa"].isin(STORM_DATES_WA)), "event_id"])

# Union-find over locality <-> event links.
parent = {}


def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    parent[find(a)] = find(b)


positives = data[data["y"] == 1]
for code_, ids in zip(positives["sal_code_2021"], positives["source_event_ids_diagnostic_only"]):
    for event in str(ids).split(";"):
        event = event.strip()
        union("L" + code_, "E" + ("STORM" if event in storm_events else event))

storm_root = find("ESTORM")
data["incident_group"] = ""
data.loc[data["y"] == 1, "incident_group"] = [
    "storm_2026_05_31" if find("L" + c) == storm_root else find("L" + c)[1:]
    for c in positives["sal_code_2021"]
]

group_sizes = data.loc[data["y"] == 1, "incident_group"].value_counts()
print(f"{len(storm_events)} storm reports merged into one incident")
print(f"{len(group_sizes)} independent incident groups; the storm group holds "
      f"{group_sizes['storm_2026_05_31']} of {data['y'].sum()} reported localities")
group_sizes.rename("reported_localities").to_frame().T

In [ ]:
data["grid_cell"] = (np.floor(data["latitude"] / GRID_DEGREES).astype(int).astype(str) + "_"
                     + np.floor(data["longitude"] / GRID_DEGREES).astype(int).astype(str))


def reference_mask(seed):
    """Unreported localities in randomly chosen grid cells, about REFERENCE_SHARE of all unreported."""
    rng = np.random.default_rng(seed)
    unlabelled = data["y"] == 0
    cells = np.array(sorted(data.loc[unlabelled, "grid_cell"].unique()))
    rng.shuffle(cells)
    target = REFERENCE_SHARE * unlabelled.sum()
    chosen, count = set(), 0
    for cell in cells:
        if count >= target:
            break
        chosen.add(cell)
        count += int((unlabelled & (data["grid_cell"] == cell)).sum())
    return unlabelled & data["grid_cell"].isin(chosen)


for s in SEEDS:
    ref = reference_mask(s)
    print(f"seed {s}: comparison set = {ref.sum()} unreported localities in "
          f"{data.loc[ref, 'grid_cell'].nunique()} grid cells")

## 4. Models compared

All models are tested on exactly the same splits.

| Model | Trained? | Why it is here |
|---|---|---|
| **Population only** | No | Ranks by 2021 population. Shows how much of the pattern is simply reporting bias. |
| **Transparent index** | No | The 50/50 hazard–sparsity score from `susceptibility_index.ipynb`, rebuilt with fire and cyclone history that ends before the outage window. |
| **Logistic regression** | Yes | Simple, interpretable baseline (log-transformed, standardised features, ridge penalty). |
| **XGBoost (original)** | Yes | First version: depth-3 trees, 300 rounds, no constraints. Kept to show why it was changed. |
| **XGBoost** | Yes | Final, regularised version: depth-2 trees, 150 rounds, stronger penalties and monotone constraints. |

The trained models weight reported localities up to balance them against the much larger unknown group. Any preprocessing is fitted on the training rows only.

**Why the XGBoost settings changed.** The original model memorised exact population values: several small localities with 112–117 residents got large boosts because a few reported localities happened to have those populations. Its trees split repeatedly at exactly those numbers. The final version makes that impossible in two ways:

- **Less flexible trees:** shallower, with more evidence needed per split and fewer rounds.
- **Monotone constraints:** more residents, a closer Optus site, or more fire or cyclone history can never *lower* the score. Site counts and densities stay unconstrained because their effect is genuinely curved.

These settings were chosen from that diagnosis, not by searching for the best test score, and are evaluated once below.

In [ ]:
def train_xgboost(X_train, y_train, seed, regularised=True):
    n_pos = int(y_train.sum())
    if regularised:
        constraints = "(" + ",".join(str(MONOTONE.get(f, 0)) for f in X_train.columns) + ")"
        params = {**XGB_PARAMS, "monotone_constraints": constraints}
        rounds = NUM_BOOST_ROUND
    else:
        params, rounds = dict(XGB_PARAMS_ORIGINAL), NUM_BOOST_ROUND_ORIGINAL
    params.update(seed=seed, scale_pos_weight=(len(y_train) - n_pos) / n_pos)
    dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(X_train.columns))
    return xgb.train(params, dtrain, num_boost_round=rounds)


def fit_predict_xgboost(X_train, y_train, X_eval, seed):
    booster = train_xgboost(X_train, y_train, seed, regularised=True)
    return booster.predict(xgb.DMatrix(X_eval, feature_names=list(X_eval.columns)))


def fit_predict_xgboost_original(X_train, y_train, X_eval, seed):
    booster = train_xgboost(X_train, y_train, seed, regularised=False)
    return booster.predict(xgb.DMatrix(X_eval, feature_names=list(X_eval.columns)))


def fit_predict_logistic(X_train, y_train, X_eval, seed, ridge=1.0, iterations=50):
    """Weighted ridge logistic regression by Newton's method (numpy only)."""
    Xt = np.log1p(X_train.to_numpy(float))
    Xe = np.log1p(X_eval.to_numpy(float))
    mean, std = Xt.mean(axis=0), Xt.std(axis=0)
    std[std == 0] = 1
    Xt = np.column_stack([np.ones(len(Xt)), (Xt - mean) / std])
    Xe = np.column_stack([np.ones(len(Xe)), (Xe - mean) / std])
    y = np.asarray(y_train, float)
    w = np.where(y == 1, (len(y) - y.sum()) / y.sum(), 1.0)
    beta = np.zeros(Xt.shape[1])
    penalty = np.eye(len(beta)) * ridge
    penalty[0, 0] = 0
    for _ in range(iterations):
        p = 1 / (1 + np.exp(-Xt @ beta))
        grad = Xt.T @ (w * (p - y)) + penalty @ beta
        hess = (Xt * (w * p * (1 - p))[:, None]).T @ Xt + penalty
        step = np.linalg.solve(hess, grad)
        beta -= step
        if np.abs(step).max() < 1e-8:
            break
    return 1 / (1 + np.exp(-Xe @ beta))


def population_only(X_train, y_train, X_eval, seed):
    return X_eval["population_2021"].to_numpy(float)


# Transparent index: same formula as susceptibility_index.ipynb (section 3),
# using fire/cyclone history that ends before the outage window.
INDEX_SPEC = {
    "hazard": [("bushfire_prone_area_pct_snapshot", "pct"), ("flood_any_mapped_extent_pct_snapshot", "pct"),
               ("recorded_bushfires_per_year_before_window", "log_high"),
               ("nearby_cyclone_track_count_before_window", "log_high")],
    "infrastructure": [("nearest_optus_site_km", "log_high"), ("optus_sites_within_5km", "log_low"),
                       ("optus_sites_within_10km", "log_low")],
}


def transparent_index(df):
    parts = {}
    for part, spec in INDEX_SPEC.items():
        scaled = []
        for col, method in spec:
            v = df[col].clip(lower=0)
            if method == "pct":
                scaled.append((v / 100).clip(0, 1))
            else:
                s = np.log1p(v) / np.log1p(v.max())
                scaled.append(s if method == "log_high" else 1 - s)
        parts[part] = pd.concat(scaled, axis=1).mean(axis=1).rank(pct=True)
    return 0.5 * parts["hazard"] + 0.5 * parts["infrastructure"]


data = data.merge(snapshot[["sal_code_2021", "bushfire_prone_area_pct_snapshot",
                            "flood_any_mapped_extent_pct_snapshot"]], on="sal_code_2021", how="left")
data["transparent_index_score"] = transparent_index(data)


def index_scores(X_train, y_train, X_eval, seed):
    return data.loc[X_eval.index, "transparent_index_score"].to_numpy(float)


MODEL_FUNCTIONS = {
    "Population only": population_only,
    "Transparent index": index_scores,
    "Logistic regression": fit_predict_logistic,
    "XGBoost (original)": fit_predict_xgboost_original,
    "XGBoost": fit_predict_xgboost,
}

## 5. Training and testing

Two tests, each repeated for every seed:

1. **Storm hold-out.** Remove the storm incident's reported localities, train on the 18 small incidents plus unknowns, then score the storm localities. *Can a model trained on other incidents find the places one big storm hit?*
2. **Leave-one-incident-out.** For each of the 18 small incidents: remove it, train on everything else (storm included), then score its localities. *Does the pattern carry over to an unseen incident?*

The comparison set is never trained on in either test. Each held-out reported locality gets a **percentile among the comparison set**: 1.0 means it scores above all of them, 0.5 means it is no better than chance.

- **Mean percentile** works like an AUC.
- **Captured in top 20%** is the share of held-out reported localities that would fall in the High band. Random ranking would capture about 20%.

In [ ]:
def percentile_among(reference_scores, scores):
    ref = np.sort(reference_scores)
    below = np.searchsorted(ref, scores, side="left")
    equal = np.searchsorted(ref, scores, side="right") - below
    return (below + 0.5 * equal) / len(ref)


small_groups = sorted(g for g in data.loc[data["y"] == 1, "incident_group"].unique() if g != "storm_2026_05_31")
SPLITS = [("Storm hold-out", ["storm_2026_05_31"])] + [("Leave-one-incident-out", [g]) for g in small_groups]

rows = []
for seed in SEEDS:
    reference = reference_mask(seed)
    for test_name, held_groups in SPLITS:
        held = data["incident_group"].isin(held_groups)
        train = ~reference & ~held
        X_train, y_train = data.loc[train, FEATURES], data.loc[train, "y"]
        X_eval = data.loc[held | reference, FEATURES]
        is_ref = reference[X_eval.index].to_numpy()
        for model_name, fn in MODEL_FUNCTIONS.items():
            scores = fn(X_train, y_train, X_eval, seed)
            pct = percentile_among(scores[is_ref], scores[~is_ref])
            for sal, p in zip(X_eval.index[~is_ref], pct):
                rows.append({"seed": seed, "test": test_name, "incident_group": held_groups[0],
                             "model": model_name, "row": sal, "percentile": p})

results = pd.DataFrame(rows)
results["captured_top_20pct"] = results["percentile"] >= 1 - HIGH_SHARE
print(f"{len(results)} held-out scores from {len(SEEDS)} seeds x {len(SPLITS)} splits x {len(MODEL_FUNCTIONS)} models")

In [ ]:
def summarise(frame):
    per_seed = frame.groupby(["model", "seed"]).agg(mean_percentile=("percentile", "mean"),
                                                   captured=("captured_top_20pct", "mean"))
    out = per_seed.groupby("model").agg(
        mean_percentile=("mean_percentile", "mean"),
        mean_percentile_min=("mean_percentile", "min"),
        mean_percentile_max=("mean_percentile", "max"),
        captured_top_20pct=("captured", "mean"),
    )
    out["lift_vs_random"] = out["captured_top_20pct"] / HIGH_SHARE
    return out.reindex(list(MODEL_FUNCTIONS))


model_order = list(MODEL_FUNCTIONS)
evaluation = {}
for name, frame in [("Storm hold-out (90 localities)", results[results["test"] == "Storm hold-out"]),
                    ("Leave-one-incident-out (24 localities, 18 incidents)",
                     results[results["test"] == "Leave-one-incident-out"]),
                    ("All 114 reported localities", results)]:
    table = summarise(frame)
    evaluation[name] = table.round(3).to_dict(orient="index")
    print(f"\n=== {name} ===")
    print(table.round(3).to_string())

In [ ]:
# Per-incident view for the small incidents (XGBoost, averaged over seeds).
loio = results[(results["test"] == "Leave-one-incident-out")]
per_incident = (loio.pivot_table(index="incident_group", columns="model", values="percentile", aggfunc="mean")
                [model_order])
per_incident.insert(0, "reported_localities", data[data["y"] == 1]["incident_group"].value_counts())
per_incident.round(2)

**How to read the results**

- **Population only** sets the reporting-bias bar. If a trained model barely beats it, the model has mostly learned "big places get reported", not network fragility.
- **The storm hold-out is the hardest test.** Weather picked which South West localities lost service, and none of the 11 features describes that storm.
- **The small incidents** are spread across WA, so they are the fairest check of whether a general pattern exists. With only 24 localities, expect wide variation.
- Scores above 0.5 show a **reporting pattern** that carries over to unseen incidents. They are **not** accuracy, because the comparison set may contain unreported outages.

## 6. Final model and what it learned

The final, regularised XGBoost model is trained on **all** 1,699 localities. Its scores for the 114 reported localities are therefore in-sample; use the tests in section 5 to judge how well it generalises.

Feature contributions are XGBoost's built-in SHAP values (`pred_contribs=True`). For each locality, they show how much each feature pushed its score up or down in log-odds.

In [ ]:
X_all, y_all = data[FEATURES], data["y"]
n_pos = int(y_all.sum())
final_model = train_xgboost(X_all, y_all, SEEDS[0], regularised=True)
d_all = xgb.DMatrix(X_all, feature_names=FEATURES)
data["pu_score"] = final_model.predict(d_all)

contribs = final_model.predict(d_all, pred_contribs=True)
contrib = pd.DataFrame(contribs[:, :-1], columns=FEATURES, index=data.index)

importance = pd.DataFrame({
    "mean_abs_contribution": contrib.abs().mean(),
    # Positive: higher feature values push the score up. Negative: higher values push it down.
    "direction_spearman": [contrib[f].corr(data[f], method="spearman") for f in FEATURES],
}).sort_values("mean_abs_contribution", ascending=False)
importance.round(3)

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib not installed; skipping chart")

if plt is not None:
    imp = importance.iloc[::-1]
    colours = ["#c8402f" if d > 0 else "#2f6fc8" for d in imp["direction_spearman"]]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(imp.index, imp["mean_abs_contribution"], color=colours)
    ax.set_xlabel("Mean |SHAP contribution| (log-odds)")
    ax.set_title("What drives the XGBoost outage-report score\nred = higher value raises score, blue = lowers it")
    plt.tight_layout()
    plt.show()

**Population-effect check.** This compares the average population contribution by population band for the original and final models. In the final model, contributions should rise steadily with population, with no isolated spike for one narrow band.

In [ ]:
original_model = train_xgboost(X_all, y_all, SEEDS[0], regularised=False)
orig_contrib = pd.DataFrame(original_model.predict(d_all, pred_contribs=True)[:, :-1], columns=FEATURES)

pop_bands = pd.cut(data["population_2021"], [-1, 0, 50, 100, 111, 117, 150, 300, 1000, 5000, 1e6],
                   labels=["0", "1-50", "51-100", "101-111", "112-117", "118-150", "151-300",
                           "301-1k", "1k-5k", "5k+"])
population_check = pd.DataFrame({
    "localities": pop_bands.value_counts(sort=False),
    "original_mean_contribution": orig_contrib["population_2021"].groupby(pop_bands, observed=False).mean(),
    "final_mean_contribution": contrib["population_2021"].groupby(pop_bands, observed=False).mean(),
})
print("Average log-odds contribution of population_2021 by population band:")
population_check.round(3)

In [ ]:
# The localities the original model boosted purely for having 112-117 residents.
spike = data["population_2021"].between(112, 117)
spike_table = pd.DataFrame({
    "locality": data.loc[spike, "locality"],
    "population_2021": data.loc[spike, "population_2021"],
    "reported": data.loc[spike, "y"],
    "original_population_contribution": orig_contrib.loc[spike, "population_2021"].round(2),
    "final_population_contribution": contrib.loc[spike, "population_2021"].round(2),
}).sort_values("original_population_contribution", ascending=False)
spike_table.head(12)

## 7. Compare with the transparent index

Model scores are banded the same way as the index (top 20% High, next 30% Medium, rest 50% Low). These are bands of **reported-outage pattern**, not susceptibility. If the two band sets disagree, the model and index are answering different questions, and that difference is the point to make in the pitch.

In [ ]:
def assign_bands(score, high_share=HIGH_SHARE, medium_share=MEDIUM_SHARE):
    rank = pd.Series(score).rank(ascending=False, method="first")
    n = len(rank)
    return pd.Series(np.where(rank <= high_share * n, "High",
                              np.where(rank <= (high_share + medium_share) * n, "Medium", "Low")),
                     index=rank.index), rank.astype(int)


data["pu_band"], data["pu_rank"] = assign_bands(data["pu_score"])

band_order = ["High", "Medium", "Low"]
index_path = PROCESSED / "wa_susceptibility_index.csv"
if index_path.exists():
    idx = pd.read_csv(index_path, dtype={"sal_code_2021": str})[["sal_code_2021", "susceptibility_band"]]
    data = data.drop(columns=["susceptibility_band"], errors="ignore").merge(idx, on="sal_code_2021", how="left")
    print("Rows: index band (current planning snapshot). Columns: XGBoost reported-outage-pattern band.")
    print(pd.crosstab(data["susceptibility_band"], data["pu_band"]).reindex(index=band_order, columns=band_order))
    print("\nSpearman correlation, index score vs XGBoost score:",
          round(data["transparent_index_score"].corr(data["pu_score"], method="spearman"), 3))
else:
    print("wa_susceptibility_index.csv not found - run susceptibility_index.ipynb first for this comparison.")

## 8. Export

- `models/xgb_pu_outage.json`: the final XGBoost model, in XGBoost's portable JSON format.
- `data/processed/wa_outage_pu_scores.csv`: score, rank, band and the top three upward contributions for each locality.
- `data/processed/wa_outage_pu_model.metadata.json`: features, parameters, test results, versions and caveats.

In [ ]:
def top_contributions(i, k=3):
    row = contrib.loc[i]
    up = row[row > 0].sort_values(ascending=False).head(k)
    return "; ".join(f"{f}={data.at[i, f]:g} (+{v:.2f})" for f, v in up.items()) or "none"


data["top_upward_contributions"] = [top_contributions(i) for i in data.index]

MODELS.mkdir(exist_ok=True)
model_path = MODELS / "xgb_pu_outage.json"
final_model.save_model(str(model_path))

scores_path = PROCESSED / "wa_outage_pu_scores.csv"
export_cols = (["sal_code_2021", "locality", "longitude", "latitude", "population_2021",
                "pu_score", "pu_rank", "pu_band", "top_upward_contributions", "y", "incident_group"]
               + (["susceptibility_band"] if "susceptibility_band" in data else []))
(data.sort_values("pu_rank")[export_cols]
 .rename(columns={"y": "reported_outage_in_window", "susceptibility_band": "index_band"})
 .to_csv(scores_path, index=False, float_format="%.6f", lineterminator="\n"))

metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "experiment": "Positive-unlabelled XGBoost on reported WA mobile outages; not a calibrated outage probability.",
    "target": {"column": TARGET, "positive": int(n_pos), "unlabelled": int(len(data) - n_pos),
               "confirmed_negative": 0},
    "features": FEATURES,
    "xgboost_params": {**XGB_PARAMS, "num_boost_round": NUM_BOOST_ROUND, "seed": SEEDS[0],
                       "monotone_constraints": MONOTONE,
                       "scale_pos_weight": round((len(y_all) - n_pos) / n_pos, 4)},
    "xgboost_params_original_for_comparison": {**XGB_PARAMS_ORIGINAL,
                                                "num_boost_round": NUM_BOOST_ROUND_ORIGINAL},
    "regularisation_reason": "Original model memorised exact population values (repeated splits at 112-117 "
                             "residents). Final model uses shallower trees, larger min_child_weight, "
                             "stronger L2, gamma, fewer rounds and monotone constraints.",
    "evaluation_design": {
        "incident_groups": int(data.loc[data["y"] == 1, "incident_group"].nunique()),
        "storm_events": sorted(storm_events),
        "tests": ["Storm hold-out", "Leave-one-incident-out over the remaining incident groups"],
        "comparison_set": f"{REFERENCE_SHARE:.0%} of unlabelled localities by {GRID_DEGREES} degree grid cell, "
                          "never trained on",
        "seeds": SEEDS,
        "metric_note": "Percentile of held-out reported localities among the comparison set (AUC-like) and "
                       "share in its top 20%. Not accuracy: the comparison set may contain unreported outages.",
    },
    "evaluation_results": evaluation,
    "feature_importance": importance.round(4).to_dict(orient="index"),
    "versions": {"python": sys.version.split()[0], "machine": platform.machine(), "numpy": np.__version__,
                 "pandas": pd.__version__, "xgboost": xgb.__version__},
    "inputs": {p.name: sha256(p) for p in INPUTS.values()},
    "outputs": {model_path.name: sha256(model_path), scores_path.name: sha256(scores_path)},
    "limitations": [
        "Positive-unlabelled data under a probably violated SCAR assumption: reporting favours populated, "
        "well-served localities.",
        "One storm accounts for most reported localities; there are few independent incidents.",
        "Final-model scores for reported localities are in-sample.",
        "Bands are relative rankings of reported-outage pattern, not susceptibility or probability.",
    ],
}
meta_path = PROCESSED / "wa_outage_pu_model.metadata.json"
meta_path.write_text(json.dumps(metadata, indent=2) + "\n")
print("Wrote", model_path.relative_to(ROOT), ",", scores_path.relative_to(ROOT), "and", meta_path.relative_to(ROOT))